In [1]:
# imports
import os
# Choose the GPU to use
# os.environ["CUDA_VISIBLE_DEVICES"] = '0'
os.environ["NCCL_DEBUG"] = "INFO"
os.environ["OMPI_MCA_opal_cuda_support"] = "true"
os.environ["CONDA_OVERRIDE_GLIBC"] = "2.56"
import sys
# sys.path.append("../../")
from collections import Counter
import datetime
import pickle
import subprocess
import seaborn as sns
sns.set()
from datasets import load_from_disk, concatenate_datasets
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import Trainer
from transformers.training_args import TrainingArguments
from genecompass import DataCollatorForCellClassification
from genecompass.utils import load_prior_embedding
import argparse
import numpy as np
import random
import torch

In [2]:
def get_model_cta(model_path, token_dictionary_path, target_name_id_dict,device = 'cuda', freeze_layers = 0):
    """
    model_path: Pretrained genecompass model path
    token_dictionary_ath: Pirior knowledge path
    taget_name_id_dict: 
    """
    if ('GeneCompass' in model_path) or (token_dictionary_path is not None):
        from genecompass import BertForSequenceClassification
        from genecompass.utils import load_prior_embedding

        # load prior knowledge embedding
        knowledges = dict()
        out = load_prior_embedding(token_dictionary_or_path=token_dictionary_path)
        knowledges['promoter'] = out[0]
        knowledges['co_exp'] = out[1]
        knowledges['gene_family'] = out[2]
        knowledges['peca_grn'] = out[3]
        knowledges['homologous_gene_human2mouse'] = out[4]

        if isinstance(target_name_id_dict, str):
            target_name_id_dict_path = target_name_id_dict
            with open(target_name_id_dict_path, 'rb') as fp:
                target_name_id_dict = pickle.load(fp)

        # reload pretrained model
        model = BertForSequenceClassification.from_pretrained(
            model_path,
            num_labels=len(target_name_id_dict.keys()),
            output_attentions=False,
            output_hidden_states=False,
            knowledges=knowledges,
        )

        if freeze_layers > 0:
            modules_to_freeze = model.bert.encoder.layer[:freeze_layers]
            for module in modules_to_freeze:
                for param in module.parameters():
                    param.requires_grad = False

        model = model.to(device)

        return model
    

def train_test_ds(ds_dir, target_name_id_dict_path):
    # data path
    train_path = os.path.join(ds_dir, 'train')
    test_path =  os.path.join(ds_dir, 'test')

    # load datasets
    train_set = load_from_disk(train_path)
    test_set = load_from_disk(test_path)

    # rename columns
    train_set = train_set.rename_column("cell_type", "label")
    test_set = test_set.rename_column("cell_type", "label")
    
    with open(target_name_id_dict_path, 'rb') as fp:
        # label 对应 cell type dict
        target_name_id_dict = pickle.load(fp)
        print(target_name_id_dict)

    # change labels to numerical ids
    def classes_to_ids(example):
        example["label"] = target_name_id_dict[example["label"]]
        return example
    train_set = train_set.map(classes_to_ids, num_proc=16)
    test_set = test_set.map(classes_to_ids, num_proc=16)

    # filter dataset for cell types in corresponding training set
    trained_labels = list(Counter(train_set['label']).keys())
    def if_trained_label(example):
        return example['label'] in trained_labels
    test_set = test_set.filter(if_trained_label, num_proc=16)

    return train_set,test_set,target_name_id_dict


In [ ]:
# set output dir
output_dir='/personal/scFMs_dynamic/results/mouse_hematopoiesis/finetune_celltype_classifer'
# make output directory
if not os.path.exists(output_dir):
    subprocess.call(f'mkdir {output_dir}', shell=True)

train_set,test_set,target_name_id_dict = train_test_ds(ds_dir='/personal/scFMs_dynamic/data/processed/Mouse_hematopoiesis_neu_mono_geneformer.dataset', target_name_id_dict_path='/share/LLM_Omics/data_ham/h5ad/geneCompass_input/liver/target_name_id_dict.pickle')
model = get_model_cta(model_path='/personal/llm_bench/genecompass_cl_output/models/genecompass_woliver_continual_learning/models', token_dictionary_path='/root/GeneCompass/prior_knowledge/human_mouse_tokens.pickle', target_name_id_dict=target_name_id_dict)


In [1]:
# 改下load dataset 函数

from datasets import load_from_disk

ds = load_from_disk('/personal/scF_dynamic/data/mouse_hematopoiesis/exp1_invitro_timecourse/gc_token.dataset')


In [2]:
ds = ds.rename_column('Cell type annotation', 'cell_type')

In [3]:
ds = ds.rename_column('Time point', 'time_point')

In [4]:
ds.save_to_disk('/personal/scFMs_dynamic/data/processed/Mouse_hematopoiesis_neu_mono_geneCompass.dataset')

Saving the dataset (0/2 shards):   0%|          | 0/34782 [00:00<?, ? examples/s]

In [8]:

ds = ds.class_encode_column('cell_type')
ds
label_names = ds.features['cell_type'].names
dict(zip(label_names, range(len(label_names))))
for i, label in enumerate(label_names):
    print(f"{i}: {label}")
train_ds = ds.train_test_split(test_size=0.2, stratify_by_column='cell_type')['train']
test_ds = ds.train_test_split(test_size=0.2, stratify_by_column='cell_type')['test']

ValueError: Column (cell_type) not in table columns (['input_ids', 'values', 'length', 'species', 'Library', 'Cell barcode', 'Time point', 'Starting population', 'Cell type annotation', 'Well', 'SPRING-x', 'SPRING-y', 'clone']).

In [2]:
ds

Dataset({
    features: ['input_ids', 'cell_type', 'time_point', 'start_population', 'clone', 'well', 'length'],
    num_rows: 34782
})